# 00 · Colab Setup & Game-API Smoke Test

**Who Wants to Be a PoliMillionaire?** — environment + game-API connectivity check.

Run this notebook first, you should — before anything else. The LLM, here we load NOT; only that the game client connects, confirm we do.

Steps: mount Drive → find `millionaire_client` → log in → list competitions. **Start a game we do NOT** — the 30-second timer, avoid it we must until the answering pipeline is ready.

> GPU you need NOT for this notebook (it tests only the API). A CPU runtime, choose — your T4 quota, save it you will.

## 1 · Mount Google Drive
Where the provided `millionaire_client` package lives (under your `NLP project` folder), this is.

In [ ]:
# Mounted, the Drive must be -- in it, the provided package lives.
from google.colab import drive
drive.mount('/content/gdrive')

## 2 · Put `millionaire_client` on the path
Auto-locate the package we do, so the exact folder nesting matters not.

In [ ]:
import sys, os, glob

# Under the 'NLP project' folder, the package we hunt -- its parent, onto sys.path it goes.
base = '/content/gdrive/MyDrive/NLP project'
hits = glob.glob(os.path.join(base, '**', 'millionaire_client', '__init__.py'), recursive=True)
assert hits, f'Found, millionaire_client was NOT, under {base} -- check the upload location/name, you must.'
pkg_parent = os.path.dirname(os.path.dirname(hits[0]))   # the parent of millionaire_client, this is
if pkg_parent not in sys.path:
    sys.path.append(pkg_parent)
print('On sys.path now:', pkg_parent)

## 3 · Log in (password from a Colab secret)
A secret named `poli-millionaire` (the 🔑 panel on the left), create first you must, and notebook access grant it. Hardcode the password, we never do.

In [ ]:
from google.colab import userdata
from millionaire_client import MillionaireClient, AuthenticationError

API_URL = 'http://131.175.15.22:51111/'

# Authenticate we must -- the password, only from the secret it comes.
client = MillionaireClient(API_URL)
try:
    user = client.login(userdata.get('username'), userdata.get('poli-millionaire'))
    print(f'Welcome, {user.username}! (role: {user.role})')
except AuthenticationError as e:
    print(f'Failed, the login did: {e}')

## 4 · List competitions (the safe smoke test)
Only list them, we do. A game we start NOT -- timed, every question is.

In [ ]:
# The competitions and their public ids (0, 1, 2, ...), these we print.
for c in client.competitions.list_all():
    print(c.id, '-', c.name, f'({c.max_levels} questions)')

---
✅ Login + a competition list, if you see — connected, the client is.

Next: share the competition ids/names, and **Phase 1** (the answering pipeline + Qwen2.5-7B 4-bit) begin we will.

⚠️ Call `client.game.start(...)` only once the pipeline is ready — the 30-second timer, immediately it begins.